### 0. Data Preview

In [ ]:
# load the npy file and preview the data
import numpy as np
import matplotlib.pyplot as plt
import pickle
import json
import os
import sys
import torch
import json
import hashlib
from sklearn.feature_extraction.text import TfidfVectorizer
from torch_geometric.utils import coalesce, to_undirected

/Users/sharkiefff/anaconda3/envs/graph/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# load from this path, name is heart_test_samples.npy
# specify the path to the npy file directly
# path = 'ogbl-ddi/heart_test_samples.npy'
# path = 'citeseer/heart_test_samples.npy'
path = '../dataset/cora/data.pkl'

data = pickle.load(open(path, 'rb'))

# check the type of loaded data
print(type(data))

# check the keys and value types in the data
for key in data.keys():
    print("============")
    print(key, type(data[key]))
    # and value shape
    if isinstance(data[key], np.ndarray):
        print(data[key].shape)
    elif isinstance(data[key], torch.Tensor):
        print(data[key].shape)
    elif isinstance(data[key], list):
        print(len(data[key]))
    elif isinstance(data[key], dict):
        print(data[key].keys())


In [4]:
# load the data from data1 folder
path = '../dataset/bluesky_repost/edge_sets.json'
bluesky_data = json.load(open(path, 'r'))

# get the keys and values
for key in bluesky_data.keys():
    print("============")
    print(key, type(bluesky_data[key]))
    # and value shape
    if isinstance(bluesky_data[key], list):
        print(len(bluesky_data[key]))
    elif isinstance(bluesky_data[key], dict):
        print(bluesky_data[key].keys())

train_edges <class 'list'>
262486
valid_edges <class 'list'>
47095
test_edges <class 'list'>
122341
node2id <class 'dict'>
dict_keys(['0000405e89cce80f', '0001610f2b58f1ad', '00039f273a60bd6f', '00040dc2b795ca5b', '000491441a6c577b', '000642e19759a07b', '000b5d8707d0508b', '000bf58cfb71cb3c', '000c3e0b9feb7f2c', '000c42719e6c1c80', '00113b6bf3a489c0', '001153488849f3f9', '0018fbb3897d70d1', '001b1e800071363e', '001c982ec3e00035', '001f54ee5988981b', '0027d56fb42aeb12', '00290d6470d5f3d2', '0029560f9a20babf', '002c861cc68b5f5e', '002fb68ab342e744', '0031de71ad0f2aa7', '003282cb31109bd8', '00373cec43adab0d', '00376e7c9c57671c', '0037a8b0ddd8bda0', '0039c7c7c7c1e92e', '003a0773c9cd9762', '003a94fb4e220c32', '003adf0348dfdc63', '003c55b8d405c61d', '003d37ff321a3854', '003fb41b2b8cc802', '0043d5d96bd6b3c1', '004596ad07e059c9', '00469087fac96041', '0046b09c1e54da13', '0047735769813f7b', '004d81c95a3e7ba6', '004f6f61d5e36815', '004fd0b9d88a8e7e', '00518640d4186bb4', '0053c3075c4e91af', '00553

### 1. Data generation (1 pos -> 1 neg)

In [ ]:

def save(file_name, data_name, edge):
    with open('../dataset/' + data_name+'/'+ file_name+ '.txt', 'w') as f:
        for i in range(edge.size(1)):
            s, t = edge[0][i].item(), edge[1][i].item()
            f.write(str(s)+'\t'+str(t) +'\n')
            f.flush()

def load_json_data(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

def generate_negatives(pos_tensor, edge_dict, nodenum):
    neg = []
    for i in range(pos_tensor.size(1)):
        src = torch.randint(0, nodenum, (1,)).item()
        dst = torch.randint(0, nodenum, (1,)).item()
        while dst in edge_dict[src] or src in edge_dict[dst]:
            src = torch.randint(0, nodenum, (1,)).item()
            dst = torch.randint(0, nodenum, (1,)).item()
        neg.append([src, dst])
    return torch.tensor(neg).t()

def generate_node_features(node2id, feature_dim=16):
    node_features = np.zeros((len(node2id), feature_dim))
    for node, idx in node2id.items():
        hash_object = hashlib.sha256(node.encode())
        hash_digest = hash_object.digest()
        feature_vector = np.frombuffer(hash_digest[:feature_dim], dtype=np.uint8)
        node_features[idx] = feature_vector
    return torch.tensor(node_features, dtype=torch.float)

# def generate_node_features(node2id, feature_dim=16):
#     vectorizer = TfidfVectorizer(max_features=feature_dim)
#     node_strings = list(node2id.keys())
#     tfidf_matrix = vectorizer.fit_transform(node_strings).toarray()
#     node_features = np.zeros((len(node2id), feature_dim))
#     for node, idx in node2id.items():
#         node_features[idx] = tfidf_matrix[node_strings.index(node)]
#     return torch.tensor(node_features, dtype=torch.float)  

In [3]:
data_name = 'bluesky_repost'
json_dir = '../dataset/bluesky_repost'
json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]

train_pos, valid_pos, test_pos = [], [], []
node_set = set()

for json_file in json_files:
    data = load_json_data(os.path.join(json_dir, json_file))
    for edge in data['train_edges']:
        train_pos.append((edge[0], edge[1]))
        node_set.add(edge[0])
        node_set.add(edge[1])
    for edge in data['valid_edges']:
        valid_pos.append((edge[0], edge[1]))
        node_set.add(edge[0])
        node_set.add(edge[1])
    for edge in data['test_edges']:
        test_pos.append((edge[0], edge[1]))
        node_set.add(edge[0])
        node_set.add(edge[1])
    # check the data['node2id] length
    print("Length of node2id: ", len(data['node2id']))
    # then save the node2id to a file
    with open(json_dir +'/node2id.txt', 'w') as f:
        for node in data['node2id']:
            f.write(node+'\t'+str(data['node2id'][node])+'\n')
            f.flush()

num_nodes = len(node_set)
print('Number of nodes: {}, Number of edges: {}'.format(num_nodes, len(train_pos) + len(valid_pos) + len(test_pos)))

Length of node2id:  120090
Number of nodes: 120090, Number of edges: 431922


In [4]:
train_pos_tensor = torch.transpose(torch.tensor(train_pos), 1, 0)
valid_pos_tensor = torch.transpose(torch.tensor(valid_pos), 1, 0)
test_pos_tensor = torch.transpose(torch.tensor(test_pos), 1, 0)

edge_index = torch.cat((train_pos_tensor, train_pos_tensor[[1, 0]]), dim=1)
edge_index = to_undirected(edge_index)
edge_index = coalesce(edge_index)

nodenum = num_nodes

edge_dict = {i: set() for i in range(nodenum)}
for i in range(edge_index.size(1)):
    s, t = edge_index[0][i].item(), edge_index[1][i].item()
    edge_dict[s].add(t)
    edge_dict[t].add(s)

valid_neg_tensor = generate_negatives(valid_pos_tensor, edge_dict, nodenum)
test_neg_tensor = generate_negatives(test_pos_tensor, edge_dict, nodenum)

node2id = data['node2id']
node_features = generate_node_features(node2id, feature_dim=16)
torch.save(node_features, '../dataset/' + data_name + '/node_features.pt')

save('train_pos', data_name, train_pos_tensor)
save('valid_pos', data_name, valid_pos_tensor)
save('valid_neg', data_name, valid_neg_tensor)
save('test_pos', data_name, test_pos_tensor)
save('test_neg', data_name, test_neg_tensor)